# Arquivo ipynb para tirar insights pontuais para entendimento aprodundado dos dados

In [1]:
import pandas as pd
from pathlib import Path
import plotly.express as px

In [2]:
path = "C:\\Users\\Tega\\Documents\\Projetos\\fiap-tech-challenge-fase3\\app\\machine-learning\\outputs\\outlier_study\\outlier_strategy_metrics.csv"

In [3]:
df = pd.read_csv(path)

In [4]:
df

,experiment,accuracy,precision,recall,f1,roc_auc,train_rows_before,train_rows_after,train_rows_removed,train_rows_removed_pct
0,baseline_keep_extremes,0.925463,0.750239,0.898684,0.817779,0.967267,4285506.0,4285506.0,0.0,0.000000
1,robust_scaler_keep_extremes,0.925352,0.749913,0.898567,0.817537,0.967220,4285506.0,4285506.0,0.0,0.000000
2,remove_train_outside_p01_p99,0.920342,0.731127,0.904690,0.808701,0.967216,4285506.0,3999570.0,285936.0,0.066722
3,winsorize_p01_p99,0.924601,0.746972,0.899598,0.816211,0.967174,4285506.0,4285506.0,0.0,0.000000
4,remove_train_outside_p05_p95,0.905826,0.685905,0.911299,0.782698,0.964646,4285506.0,2801585.0,1483921.0,0.346265
5,winsorize_p05_p95,0.916694,0.724461,0.891428,0.799318,0.964276,4285506.0,4285506.0,0.0,0.000000


In [3]:
base_dir = Path('../database/')
input_file_path = base_dir.parent / "database"

In [ ]:
df_outliers = pd.read_csv(input_file_path / "outliers.csv")

In [4]:
#df_routes = pd.read_csv(input_file_path / "outputs/completed_route_delay_profile.csv")
df_complete = pd.read_csv(input_file_path / "flights_treated.csv")

In [ ]:
#df_routes["ROUTE"].nunique()

In [ ]:
df_complete.head(10)

## Entendendo o impacto se eu tratar o p99 da base;

In [4]:
df_complete = pd.read_csv(input_file_path / "flights_treated.csv")

In [10]:
df_complete.shape[0]

5714008

In [8]:
df_complete[df_complete["IS_DELAYED_15"] == True].count()

YEAR                        1063439
MONTH                       1063439
DAY                         1063439
DAY_OF_WEEK                 1063439
AIRLINE                     1063439
FLIGHT_NUMBER               1063439
TAIL_NUMBER                 1063439
ORIGIN_AIRPORT              1063439
DESTINATION_AIRPORT         1063439
SCHEDULED_DEPARTURE         1063439
DEPARTURE_TIME              1063439
DEPARTURE_DELAY             1063439
TAXI_OUT                    1063439
WHEELS_OFF                  1063439
SCHEDULED_TIME              1063439
ELAPSED_TIME                1063439
AIR_TIME                    1063439
DISTANCE                    1063439
WHEELS_ON                   1063439
TAXI_IN                     1063439
SCHEDULED_ARRIVAL           1063439
ARRIVAL_TIME                1063439
ARRIVAL_DELAY               1063439
AIR_SYSTEM_DELAY            1063439
SECURITY_DELAY              1063439
AIRLINE_DELAY               1063439
LATE_AIRCRAFT_DELAY         1063439
WEATHER_DELAY               

In [5]:
# Analisando outliers
numeric_cols = df_complete.select_dtypes(include="number").columns
numeric_profile = df_complete[numeric_cols].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T
numeric_profile["iqr"] = numeric_profile["75%"] - numeric_profile["25%"]
numeric_profile["lower_iqr_limit"] = numeric_profile["25%"] - 1.5 * numeric_profile["iqr"]
numeric_profile["upper_iqr_limit"] = numeric_profile["75%"] + 1.5 * numeric_profile["iqr"]
numeric_profile["outlier_count_iqr"] = [
    (
        (df_complete[col] < numeric_profile.loc[col, "lower_iqr_limit"])
        | (df_complete[col] > numeric_profile.loc[col, "upper_iqr_limit"])
    ).sum()
    for col in numeric_profile.index
]
numeric_profile["outlier_pct_iqr"] = (
    numeric_profile["outlier_count_iqr"] / len(df_complete) * 100
).round(2)

numeric_profile

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,iqr,lower_iqr_limit,upper_iqr_limit,outlier_count_iqr,outlier_pct_iqr
YEAR,5714008.0,2015.000000,0.000000,2015.00000,2015.00000,2015.00000,2015.00000,2015.00000,2015.00000,2015.00000,2015.00000,2015.00000,0.00000,2015.000000,2015.000000,0,0.00
MONTH,5714008.0,6.547799,3.397421,1.00000,1.00000,1.00000,4.00000,7.00000,9.00000,12.00000,12.00000,12.00000,5.00000,-3.500000,16.500000,0,0.00
DAY,5714008.0,15.707591,8.774394,1.00000,1.00000,2.00000,8.00000,16.00000,23.00000,29.00000,31.00000,31.00000,15.00000,-14.500000,45.500000,0,0.00
DAY_OF_WEEK,5714008.0,3.932643,1.985967,1.00000,1.00000,1.00000,2.00000,4.00000,6.00000,7.00000,7.00000,7.00000,4.00000,-4.000000,12.000000,0,0.00
FLIGHT_NUMBER,5714008.0,2164.383547,1754.706022,1.00000,29.00000,166.00000,728.00000,1681.00000,3211.00000,5562.00000,6460.00000,9320.00000,2483.00000,-2996.500000,6935.500000,26659,0.47
SCHEDULED_DEPARTURE,5714008.0,1328.907421,483.525058,1.00000,530.00000,610.00000,916.00000,1325.00000,1730.00000,2115.00000,2245.00000,2359.00000,814.00000,-305.000000,2951.000000,0,0.00
DEPARTURE_TIME,5714008.0,1335.065627,496.419817,1.00000,511.00000,606.00000,921.00000,1330.00000,1740.00000,2132.00000,2300.00000,2400.00000,819.00000,-307.500000,2968.500000,0,0.00
DEPARTURE_DELAY,5714008.0,9.294842,36.889724,-82.00000,-13.00000,-9.00000,-5.00000,-2.00000,7.00000,67.00000,167.00000,1988.00000,12.00000,-23.000000,25.000000,730519,12.78
TAXI_OUT,5714008.0,16.065498,8.882449,1.00000,6.00000,8.00000,11.00000,14.00000,19.00000,31.00000,50.00000,225.00000,8.00000,-1.000000,31.000000,281167,4.92
WHEELS_OFF,5714008.0,1357.099048,498.023745,1.00000,510.00000,620.00000,935.00000,1343.00000,1754.00000,2143.00000,2307.00000,2400.00000,819.00000,-293.500000,2982.500000,0,0.00
